# Google Mobility EDA (Vietnam)

#### Duy Nguyen
#### DATA 5310 Data Visualization, Fall Quarter 2026

My dataset for the group project. Quick draft EDA: load it, poke around, check nulls and anything weird, clean it up, and make a couple of quick plots before trusting it for real charts. (The longer version is Part 2 of my HW8 notebook.)

In [13]:
import io
import os
import unicodedata
import urllib.request
import zipfile

import altair as alt
import pandas as pd

_ = alt.data_transformers.disable_max_rows()   # the national data is ~6,000 rows once it's long format

## Where the data comes from

**Source:** Google LLC, "Google COVID-19 Community Mobility Reports", https://www.google.com/covid19/mobility/ (accessed Nov 11, 2026). Google has one big zip with every country by year; I only use the 3 Vietnam files (2020, 2021, 2022). Google stopped updating these on Oct 15, 2022.

**One row =** one place on one day. The place is either Vietnam as a whole or one of the 63 provinces/cities.

**The columns that matter:** six measures, each a % change vs a baseline:
- retail & recreation (restaurants, cafes, shopping centers)
- grocery & pharmacy
- parks
- transit stations
- workplaces
- residential (time spent at home, so it moves way less than the others)

The baseline is the median for the same weekday during Jan 3 – Feb 6, 2020. It's also only people with Google Location History turned on, so it's a sample, not everyone.

**Google's warnings** ([docs](https://www.google.com/covid19/mobility/data_documentation.html)): don't compare countries, or cities vs rural areas; don't compare single days (especially weekday vs weekend); don't compare residential with the other five; treat gaps as unknown; holidays can look like a COVID response; and the baseline doesn't account for seasons.

## Load the data

In [14]:
# use my downloaded files if they're in a data/ folder, otherwise download google's zip
# (about 98 MB) and only pull the 3 vietnam files out of it
YEARS = (2020, 2021, 2022)
local_files = [f'data/{year}_VN_Region_Mobility_Report.csv' for year in YEARS]
if all(os.path.exists(f) for f in local_files):
    mob = pd.concat([pd.read_csv(f, parse_dates=['date']) for f in local_files], ignore_index=True)
    print('read the local files:', local_files)
else:
    url = 'https://www.gstatic.com/covid19/mobility/Region_Mobility_Report_CSVs.zip'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})) as r:
        google_zip = zipfile.ZipFile(io.BytesIO(r.read()))
    vn_files = sorted(n for n in google_zip.namelist() if n.endswith('_VN_Region_Mobility_Report.csv'))
    mob = pd.concat([pd.read_csv(google_zip.open(n), parse_dates=['date']) for n in vn_files],
                    ignore_index=True)
    print('downloaded from google:', vn_files)

print(f'{mob.shape[0]:,} rows x {mob.shape[1]} columns, '
      f'{mob["date"].min():%b %d, %Y} to {mob["date"].max():%b %d, %Y}')
mob.sample(5, random_state=5310)

read the local files: ['data/2020_VN_Region_Mobility_Report.csv', 'data/2021_VN_Region_Mobility_Report.csv', 'data/2022_VN_Region_Mobility_Report.csv']
62,308 rows x 15 columns, Feb 15, 2020 to Oct 15, 2022


,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
3844,VN,Vietnam,Ca Mau,NaN,NaN,VN-59,NaN,ChIJez-heqBJoTERs-aysVPn-a8,2020-02-22,-15.0,-15.0,-10.0,-21.0,-16.0,7.0
33882,VN,Vietnam,Lam Dong,NaN,NaN,VN-35,NaN,ChIJfdvzWSYTcTERbPVUX9_SWrs,2021-08-15,-65.0,-35.0,-55.0,-76.0,-36.0,8.0
4077,VN,Vietnam,Ca Mau,NaN,NaN,VN-59,NaN,ChIJez-heqBJoTERs-aysVPn-a8,2020-10-12,-33.0,-12.0,-33.0,-24.0,-2.0,3.0
17085,VN,Vietnam,Tây Ninh,NaN,NaN,VN-37,NaN,ChIJMXJO1JFCCzERzWzchde8giQ,2020-05-25,-13.0,10.0,-62.0,15.0,24.0,3.0
59122,VN,Vietnam,Son La,NaN,NaN,VN-05,NaN,ChIJ1R093JBZMjERezgPalpkLVg,2022-09-28,12.0,50.0,23.0,-4.0,NaN,12.0


## Check the data

In [15]:
# column types and how many non-null values each column has
mob.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62308 entries, 0 to 62307
Data columns (total 15 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   country_region_code                                 62308 non-null  object        
 1   country_region                                      62308 non-null  object        
 2   sub_region_1                                        61334 non-null  object        
 3   sub_region_2                                        0 non-null      float64       
 4   metro_area                                          0 non-null      float64       
 5   iso_3166_2_code                                     61334 non-null  object        
 6   census_fips_code                                    0 non-null      float64       
 7   place_id                                            62308 non-null  object        
 8   date  

In [16]:
# summary stats for the six measures (% change from baseline), all places and days together
mob.filter(like='_percent_change_from_baseline').describe().round(1)

,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
count,61632.0,58970.0,61037.0,56502.0,38982.0,61941.0
mean,-14.1,3.3,-7.7,-19.0,-9.5,-0.8
std,24.4,26.9,35.3,30.1,24.2,10.6
min,-97.0,-97.0,-96.0,-97.0,-93.0,-46.0
25%,-26.0,-12.0,-30.0,-39.0,-21.0,-10.0
50%,-14.0,2.0,-15.0,-19.0,-6.0,1.0
75%,-1.0,18.0,10.0,-2.0,7.0,7.0
max,145.0,268.0,361.0,197.0,62.0,56.0


In [17]:
# which columns are empty or only have one value, and how many rows are the whole country
# (no province name) vs the provinces
national_rows = mob['sub_region_1'].isna()
print('empty columns:     ', [c for c in mob.columns if mob[c].isna().all()])
print('one value only:    ', [c for c in mob.columns if mob[c].nunique() == 1])
print(f'national rows:      {national_rows.sum()} (one per day)')
print(f'province rows:      {(~national_rows).sum():,} for {mob["sub_region_1"].nunique()} provinces '
      f'and {mob["iso_3166_2_code"].nunique()} province codes')
print('repeated place-days:', mob.duplicated(['sub_region_1', 'date']).sum())

# some provinces are missing whole days (no row at all)
n_days = mob['date'].nunique()
rows_per_province = mob[~national_rows].groupby('sub_region_1').size()
short = (n_days - rows_per_province)[lambda gap: gap > 0]
print(f'province-days with no row: {short.sum()} ({short.to_dict()})')

# province names are written different ways: with accents, without, some with 'Province' on the end
names = pd.Series(mob['sub_region_1'].dropna().unique())
print('accented names:    ', names[~names.map(str.isascii)].head(5).tolist())
print("'Province' suffix: ", names[names.str.endswith(' Province')].tolist())

empty columns:      ['sub_region_2', 'metro_area', 'census_fips_code']
one value only:     ['country_region_code', 'country_region']
national rows:      974 (one per day)
province rows:      61,334 for 63 provinces and 63 province codes
repeated place-days: 0
province-days with no row: 28 ({'Bắc Kạn Province': 15, 'Dien Bien': 5, 'Ha Giang': 1, 'Lai Chau': 7})
accented names:     ['Bắc Kạn Province', 'Bình Định', 'Bình Thuận', 'Cần Thơ', 'Hà Nam']
'Province' suffix:  ['Bắc Kạn Province', 'Bac Ninh Province', 'Phu Tho Province', 'Quang Binh Province']


In [18]:
# % of rows with no value, by year, for the national row and for the provinces
categories = [c for c in mob.columns if c.endswith('_percent_change_from_baseline')]
missing = (mob.assign(year=mob['date'].dt.year,
                      level=national_rows.map({True: 'national', False: 'provinces'}))
              .groupby(['level', 'year'])[categories]
              .apply(lambda rows: rows.isna().mean().mul(100).round(1))
              .rename(columns=lambda c: c.replace('_percent_change_from_baseline', '')))
last_workplaces = mob.loc[mob['workplaces_percent_change_from_baseline'].notna(), 'date'].max()
print(f'last day with a workplaces value: {last_workplaces:%b %d, %Y}')
missing

last day with a workplaces value: Oct 19, 2021


retail_and_recreation  grocery_and_pharmacy  parks  \
level     year                                                       
national  2020                    0.0                   0.0    0.0   
          2021                    0.0                   0.0    0.0   
          2022                    0.0                   0.0    0.0   
provinces 2020                    3.3                   7.9    3.9   
          2021                    0.0                   4.1    0.2   
          2022                    0.0                   4.5    2.4   

                transit_stations  workplaces  residential  
level     year                                             
national  2020               0.0         0.0          0.0  
          2021               0.0        20.0          0.0  
          2022               0.0       100.0          0.0  
provinces 2020              13.0         1.1          1.5  
          2021               7.4        20.0          0.1  
          2022               8.1       100.0          0.2

In [19]:
# check 1: Tet (Lunar New Year) looks like a lockdown. national values in the week around
# Lunar New Year 2021 (Feb 12) vs the week before
nat_raw = mob[national_rows].set_index('date')[categories].rename(
    columns=lambda c: c.replace('_percent_change_from_baseline', ''))
pd.DataFrame({'Feb 1-7, 2021': nat_raw.loc['2021-02-01':'2021-02-07'].mean(),
              'Feb 10-16, 2021 (Tet)': nat_raw.loc['2021-02-10':'2021-02-16'].mean()}).round(1)

,"Feb 1-7, 2021","Feb 10-16, 2021 (Tet)"
retail_and_recreation,-12.3,-25.7
grocery_and_pharmacy,14.6,1.9
parks,-18.1,20.7
transit_stations,-12.4,-37.3
workplaces,-10.7,-69.3
residential,-8.0,-3.9


In [20]:
# check 2: is 0 actually 'normal' for vietnam? average for a calm stretch with no lockdown and
# few cases (Oct to Dec 2020)
nat_raw.loc['2020-10-01':'2020-12-31'].mean().round(1).to_frame('Oct-Dec 2020 average')

,Oct-Dec 2020 average
retail_and_recreation,-15.4
grocery_and_pharmacy,2.5
parks,-22.1
transit_stations,-6.4
workplaces,6.6
residential,-5.7


**What I found**
- The national row is basically complete. Only workplaces has gaps, and it stops completely after Oct 19, 2021.
- All 63 provinces are there and nothing is duplicated, but 28 province-days are just missing (Bac Kan 15, Lai Chau 7, Dien Bien 5, Ha Giang 1).
- 3 columns are totally empty.
- Province names are messy (some with accents, some without, 4 end in "Province"), so I match on the ISO code instead.
- The province rows have some gaps where Google didn't have enough data, mostly in transit and grocery.

Two things to watch out for:
- **Tết looks like a lockdown.** In the Lunar New Year week of 2021, workplace visits were 69% below baseline vs 11% the week before. That's the holiday, not COVID rules, so I'll mark Tết on charts.
- **0 isn't "normal" for Vietnam.** Even in calm late 2020, shop visits were ~15% below baseline and time at home ~6% below. The baseline weeks include the Tết 2020 rush and the holiday, so I compare against the calm late-2020 level instead of 0.

## Clean the data

- drop the empty / one-value columns
- shorten the six column names (e.g. `retail_and_recreation_percent_change_from_baseline` → `retail`)
- one plain name per province (no accents, no "Province"), checked against the ISO code; the national row becomes "Vietnam"
- leave gaps as missing, like Google says
- add a 7-day average for plots, since single days jump around between weekdays and weekends

In [21]:
def plain_name(name):
    # 'Bắc Kạn Province' -> 'Bac Kan'
    name = name.removesuffix(' Province').replace('Đ', 'D').replace('đ', 'd')
    return ''.join(ch for ch in unicodedata.normalize('NFKD', name) if not unicodedata.combining(ch))


short_names = {'retail_and_recreation_percent_change_from_baseline': 'retail',
               'grocery_and_pharmacy_percent_change_from_baseline': 'grocery',
               'parks_percent_change_from_baseline': 'parks',
               'transit_stations_percent_change_from_baseline': 'transit',
               'workplaces_percent_change_from_baseline': 'workplaces',
               'residential_percent_change_from_baseline': 'residential'}
MEASURES = list(short_names.values())

mob_clean = (mob.rename(columns=short_names)
                .assign(place=lambda d: d['sub_region_1'].map(plain_name, na_action='ignore').fillna('Vietnam'),
                        code=lambda d: d['iso_3166_2_code'].fillna('VN'))
                [['place', 'code', 'date'] + MEASURES])

# every code should have exactly one name and every name one code
print(f'{mob_clean["place"].nunique()} places (Vietnam + provinces); '
      f'one name per code: {(mob_clean.groupby("code")["place"].nunique() == 1).all()}; '
      f'one code per name: {(mob_clean.groupby("place")["code"].nunique() == 1).all()}')

# vietnam as a whole, smoothed with a centered 7-day average, plus the calm late-2020 level
nat = mob_clean[mob_clean['place'] == 'Vietnam'].set_index('date')[MEASURES]
nat_7d = nat.rolling(7, center=True, min_periods=7).mean()
calm_nat = nat.loc['2020-10-01':'2020-12-31'].mean()
mob_clean.sample(5, random_state=5310)

64 places (Vietnam + provinces); one name per code: True; one code per name: True


,place,code,date,retail,grocery,parks,transit,workplaces,residential
3844,Ca Mau,VN-59,2020-02-22,-15.0,-15.0,-10.0,-21.0,-16.0,7.0
33882,Lam Dong,VN-35,2021-08-15,-65.0,-35.0,-55.0,-76.0,-36.0,8.0
4077,Ca Mau,VN-59,2020-10-12,-33.0,-12.0,-33.0,-24.0,-2.0,3.0
17085,Tay Ninh,VN-37,2020-05-25,-13.0,10.0,-62.0,15.0,24.0,3.0
59122,Son La,VN-05,2022-09-28,12.0,50.0,23.0,-4.0,NaN,12.0


## Quick plots

In [22]:
# quick look at all six measures for vietnam as a whole (7-day average). the five visit measures
# share one scale; residential (time at home) gets its own because google says not to compare it
long = nat_7d.reset_index().melt('date', var_name='measure', value_name='change')
visits = alt.Chart(long[long['measure'] != 'residential']).mark_line(color='#333333', strokeWidth=1).encode(
    x=alt.X('date:T', title=None),
    y=alt.Y('change:Q', title=None),
    row=alt.Row('measure:N', title=None, sort=MEASURES[:-1], header=alt.Header(labelAngle=0, labelAlign='left'))
).properties(width=620, height=70)
home = alt.Chart(long[long['measure'] == 'residential']).mark_line(color='#333333', strokeWidth=1).encode(
    x=alt.X('date:T', title=None),
    y=alt.Y('change:Q', title=None)
).properties(width=620, height=70, title='residential = time at home (own scale)')
alt.vconcat(visits, home).properties(title=alt.TitleParams(
    'Every kind of trip fell in April 2020 and summer 2021, and time at home went up',
    subtitle='Vietnam as a whole, % change from Google\'s baseline, centered 7-day average'))

alt.VConcatChart(...)

In [23]:
# how far each province's shop and restaurant visits fell in the summer 2021 lockdown
# (Jul 19 - Sep 30, 2021) compared with its own calm late-2020 level
calm_retail = mob_clean[mob_clean['date'].between('2020-10-01', '2020-12-31')].groupby('place')['retail'].mean()
lockdown_retail = mob_clean[mob_clean['date'].between('2021-07-19', '2021-09-30')].groupby('place')['retail'].mean()
province_change = ((lockdown_retail - calm_retail).drop('Vietnam').round(1)
                   .rename('change').rename_axis('place').reset_index())

# provinces under a Directive 16 stay-home order for at least part of that window: the 19
# southern provinces on Tuoi Tre News's Jul 17, 2021 list, plus Hanoi (Jul 24), Da Nang (Jul 31)
# and Khanh Hoa (Aug 6). sources are in my HW8 notebook (Part 3)
STAY_HOME_ORDER = ['Ho Chi Minh City', 'Binh Duong', 'Dong Nai', 'Can Tho', 'Binh Phuoc',
                   'Tay Ninh', 'Ba Ria - Vung Tau', 'Tien Giang', 'Long An', 'Vinh Long',
                   'Dong Thap', 'Ben Tre', 'Hau Giang', 'An Giang', 'Bac Lieu', 'Soc Trang',
                   'Tra Vinh', 'Ca Mau', 'Kien Giang', 'Hanoi', 'Da Nang', 'Khanh Hoa']
province_change['under_order'] = province_change['place'].isin(STAY_HOME_ORDER)
print('all 22 names found:', province_change['under_order'].sum() == len(STAY_HOME_ORDER))

groups = province_change.groupby('under_order')['change'].agg(['count', 'min', 'median', 'max'])
groups.index = groups.index.map({True: 'under a stay-home order', False: 'other provinces'})
print(groups.round(1), '\n')
print('the 24 biggest drops:')
print(province_change.nsmallest(24, 'change').to_string(index=False))

all 22 names found: True
                         count   min  median   max
under_order                                       
other provinces             41 -44.3   -12.6   8.9
under a stay-home order     22 -81.3   -59.1 -51.9 

the 24 biggest drops:
            place  change  under_order
       Binh Duong   -81.3         True
          Can Tho   -80.2         True
 Ho Chi Minh City   -78.1         True
Ba Ria - Vung Tau   -76.2         True
         Dong Nai   -74.3         True
            Hanoi   -68.6         True
          Long An   -67.0         True
        Dong Thap   -63.7         True
       Tien Giang   -62.4         True
        Vinh Long   -61.0         True
         Tra Vinh   -59.4         True
         An Giang   -58.8         True
        Soc Trang   -56.0         True
         Tay Ninh   -55.4         True
       Kien Giang   -54.9         True
       Binh Phuoc   -54.7         True
          Ben Tre   -54.3         True
           Ca Mau   -54.3         True
      

In [24]:
# one dot per province, sorted, filled = under a stay-home order
alt.Chart(province_change).mark_point(size=40, color='#333333').encode(
    x=alt.X('change:Q', title='Change in shop and restaurant visits (points), summer 2021 minus late 2020'),
    y=alt.Y('place:N', title=None, sort='x'),
    fill=alt.condition('datum.under_order', alt.value('#333333'), alt.value('white')),
).properties(width=420, height=63 * 12,
             title='Shop visits fell most in the provinces under a stay-home order (filled dots)')

alt.Chart(...)

## Summary (draft)

**What stands out**
- Movement dropped hard twice: April 2020 (nationwide stay-home order) and summer 2021 (Delta wave, worst Jul–Sep). Every kind of trip fell and time at home went up.
- In between it stayed near the late-2020 level (except Da Nang's lockdown in summer 2020 and Tết). By mid-2022 shop and park visits were above Google's baseline, transit still below.
- Summer 2021 was regional: the 22 provinces I found under a stay-home order had the 22 biggest drops in shop visits (52–81 points), the next one only 44, and lots of northern/central provinces barely moved. Google says don't compare cities vs rural areas, but both groups have both, so I'm only leaning on the gap between the groups.
- The orders went where the outbreak was worst, so this data alone can't tell the rules apart from fear of the virus.

**Does it need more data?** Yes. Google only shows what people did, not what the rules were or how bad the outbreak was, so it needs the stringency index + case counts from OWID (that's Plot 1). There's no dataset of order dates by province, so I put that list together by hand from news reports. For the group: Google has India, the US and South Korea too, but Google says don't compare countries, so each country should be compared with its own calm period.

**Chart ideas:** rules + cases + shop visits on one time axis (Plot 1), a cleaner version of the province dot plot, and visits the week before vs the first week of each stay-home order.

## Decisions from this EDA

| Decision | Why |
|---|---|
| Use retail & recreation (shop visits) as the main measure | complete every day; workplaces stops Oct 2021; time at home barely moves |
| Compare to the calm late-2020 level, not 0 | Google's baseline includes Tết, so 0 isn't normal |
| Mark Tết on charts | the holiday looks like a lockdown |
| Use 7-day averages, not single days | weekdays vs weekends jump around |
| National row for trends, provinces only for the regional comparison | province rows have gaps |
| Match provinces by ISO code | names are spelled different ways |
| Leave gaps as missing | Google says gaps are unknown |
| Compare groups of provinces, not single ones | Google warns about comparing cities vs rural areas |
| Add OWID rules + cases | Google shows what people did, not why |
| Compare each country with its own calm period | Google says don't compare countries |